In [1]:
import torch
import numpy as np
import pandas as pd
import transformers
import sys
import os
src_path = os.path.abspath(os.path.join(os.getcwd(), '..', ''))
sys.path.append(src_path)
from transformers import BertTokenizer
# evaluation
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np
import regex as re

class SBERTDataset(Dataset):
    def __init__(self, dataframe, use_normalized_score=True):
        # convert to list
        self.reference_answers = dataframe['reference_answer'].tolist()
        self.student_answers = dataframe['answer'].tolist() 
        
        # Use either normalized or raw score based on parameter
        if use_normalized_score:
            self.scores = dataframe['normalized_score'].values.astype(np.float32)
        else:
            self.scores = dataframe['score'].values.astype(np.float32)
    
    def preprocess_text(self, text):
        # Remove extra whitespace
        text = ' '.join(text.split())
        # Convert to lowercase
        text = text.lower()
        # Remove special characters (keep punctuation)
        text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
        return text
    
    def __len__(self):
        return len(self.scores)
    
    def __getitem__(self, idx):
        return {
            'reference_answer': self.preprocess_text(self.reference_answers[idx]),
            'student_answer': self.preprocess_text(self.student_answers[idx]),
            'score': torch.tensor(self.scores[idx], dtype=torch.float)
        }

In [3]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F

class SiameseModel(nn.Module):
    def __init__(self, model_name='sentence-transformers/paraphrase-multilingual-mpnet-base-v2', dropout=0.1):
        super(SiameseModel, self).__init__()
        
        # Load the model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        
        # Get embedding dimension from the model config
        self.embedding_dim = self.encoder.config.hidden_size
    
    def mean_pooling(self, model_output, attention_mask):
        # Mean pooling - take average of all token embeddings
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    def get_embeddings(self, texts):
        # Tokenize the input texts
        encoded_input = self.tokenizer(
            texts, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors='pt'
        )
        
        # Move to the same device as the model
        device = next(self.parameters()).device
        encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
        
        # Get model output (without torch.no_grad to allow fine-tuning)
        outputs = self.encoder(**encoded_input)
        
        # Apply mean pooling to get sentence embeddings
        embeddings = self.mean_pooling(outputs, encoded_input['attention_mask'])
        return embeddings
    
    def forward(self, reference_texts, student_texts, sim_type='cosine'):
        # Get embeddings
        reference_embeddings = self.get_embeddings(reference_texts)
        student_embeddings = self.get_embeddings(student_texts)
        
        # Normalize embeddings
        ref_embedding = F.normalize(reference_embeddings, p=2, dim=1)
        student_embedding = F.normalize(student_embeddings, p=2, dim=1)
        
        if sim_type == 'cosine':
            # Compute cosine similarity
            similarity = torch.sum(ref_embedding * student_embedding, dim=1).unsqueeze(1)
        elif sim_type == 'manhattan':
            # Manhattan distance similarity
            manhattan_distance = torch.sum(torch.abs(ref_embedding - student_embedding), dim=1)
            similarity = (1 / (1 + manhattan_distance)).unsqueeze(1)
        elif sim_type == 'euclidean':
            # Euclidean distance similarity
            euclidean_distance = torch.sqrt(torch.sum((ref_embedding - student_embedding) ** 2, dim=1))
            similarity = (1 / (1 + euclidean_distance)).unsqueeze(1)
        
        return similarity

In [4]:
class inference:
    def __init__(self, df, model_name, model_dir, MODEL_CLASS, dropout = 0.1, model_type='bert'):
        self.model = MODEL_CLASS(model_name, dropout).to(device)
        self.model_type = model_type
        if(model_name=="indobenchmark/indobert-lite-base-p2"):
            self.tokenizer = BertTokenizer.from_pretrained(model_name)
        else:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        if(model_dir != ""):
            self.model.load_state_dict(torch.load(model_dir, weights_only=True))
        
        self.dataset= SBERTDataset(df)

    def get_prediction(self, index):
        self.model.eval()
        data = self.dataset[index]
        label = round(data['score'].item(),2)
        reference_answer = data['reference_answer']
        student_answer = data['student_answer']
        with torch.no_grad():
            predictions = self.model(reference_answer, student_answer)

        predicted_score = round(predictions.squeeze().item(), 2)
        return label, predicted_score

In [5]:
df = pd.read_csv("../../../data/aes_dataset_indo_unseen.csv")
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   question          219 non-null    object 
 1   reference_answer  219 non-null    object 
 2   answer            219 non-null    object 
 3   score             219 non-null    float64
 4   normalized_score  219 non-null    float64
 5   dataset           219 non-null    object 
 6   dataset_num       219 non-null    object 
dtypes: float64(2), object(5)
memory usage: 12.1+ KB
None


,question,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [6]:
transformers.utils.logging.set_verbosity_error()
model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
model_dir = "../../../experiments/models/best_model/sbert_cos_sim1.pt"
pipeline = inference(df, model_name, model_dir, SiameseModel)
pretrained_pipeline = inference(df, model_name, model_dir="", MODEL_CLASS=SiameseModel)

In [7]:
df['sbert_predicted'] = df.apply(lambda x: pipeline.get_prediction(x.name)[1], axis=1)
df['pre_sbert_predicted'] = df.apply(lambda x: pretrained_pipeline.get_prediction(x.name)[1], axis=1)

c:\Users\User\Documents\Code\env\lib\site-packages\transformers\models\xlm_roberta\modeling_xlm_roberta.py:371: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [8]:
results = []

for col in df.columns[7:]:
    mse = mean_squared_error(df['normalized_score'], df[col])
    rmse = np.sqrt(mse)
    pearson_corr, _ = pearsonr(df['normalized_score'], df[col])
    
    # Append results as a dictionary
    results.append({'Predicted': col, 'dataset': 'indo unseen', 'MSE': mse, 'RMSE': rmse, 'Pearson Correlation': pearson_corr})

results_df = pd.DataFrame(results).set_index('Predicted')
results_df.head(10)

,dataset,MSE,RMSE,Pearson Correlation
Predicted,,,,
sbert_predicted,indo unseen,0.073067,0.270308,0.646366
pre_sbert_predicted,indo unseen,0.179753,0.423973,0.507655


## SYNTHETIC DATA

In [9]:
data = {
    "reference_answer" : [
        "Fungsi karbohidrat adalah sebagai pemasok energi, dapat memperlancar proses pada pencernaan, memberikan efek kenyang dengan kandungan selulosa-nya dan penyeimbang asam dan basa dalam tubuh"
    ]*12,
    "answer": [
        "Karbohidrat adalah sumber energi utama tubuh.",
        "Fungsi karbohidrat adalah sebagai sumber energi dan membantu memperlancar pencernaan.",
        "Karbohidrat memberikan energi untuk aktivitas harian serta memperlancar proses pencernaan.",
        "Selain sebagai sumber energi, karbohidrat juga membantu menjaga keseimbangan asam-basa dan memberikan rasa kenyang melalui seratnya.",
        "Karbohidrat berperan sebagai sumber energi, memperlancar pencernaan, memberikan rasa kenyang berkat seratnya, dan menyeimbangkan asam dan basa dalam tubuh.",
        "Karbohidrat menyediakan glukosa yang mendukung metabolisme tubuh dan memastikan proses pencernaan berjalan lancar.",
        "Karbohidrat memberikan tenaga, membantu pencernaan, dan menjaga kestabilan pH tubuh sebagai penyeimbang asam-basa.",
        "Fungsi karbohidrat meliputi penyediaan energi, dukungan pada metabolisme dan pencernaan, serta memberikan rasa kenyang dan menjaga keseimbangan asam-basa.",
        "Sebagai sumber energi utama, karbohidrat membantu pencernaan dan menciptakan rasa kenyang karena kandungan serat, serta berperan dalam menjaga keseimbangan pH tubuh.",
        "Karbohidrat berfungsi ganda: menyediakan energi yang dibutuhkan untuk aktivitas sehari-hari dan mendukung proses pencernaan; selain itu, melalui serat yang terkandung, mereka membantu menciptakan rasa kenyang serta mengatur keseimbangan asam dan basa dalam tubuh.",
        "memberikan nutrisi",
        "untuk nutrisi tubuh"
    ],
    "normalized_score": [
        0.18,
        0.22,
        0.24,
        0.47,
        0.80,
        0.30,
        0.35,
        0.75,
        0.82,
        0.85,
        0.03,
        0.03
    ]
}

new_data = pd.DataFrame(data)
new_data.head()

,reference_answer,answer,normalized_score
0,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat adalah sumber energi utama tubuh.,0.18
1,Fungsi karbohidrat adalah sebagai pemasok ener...,Fungsi karbohidrat adalah sebagai sumber energ...,0.22
2,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat memberikan energi untuk aktivitas ...,0.24
3,Fungsi karbohidrat adalah sebagai pemasok ener...,"Selain sebagai sumber energi, karbohidrat juga...",0.47
4,Fungsi karbohidrat adalah sebagai pemasok ener...,"Karbohidrat berperan sebagai sumber energi, me...",0.80


In [10]:
transformers.utils.logging.set_verbosity_error()
model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
model_dir = "../../../experiments/models/best_model/sbert_cos_sim1.pt"
pipeline = inference(new_data, model_name, model_dir, SiameseModel)
pretrained_pipeline = inference(new_data, model_name, model_dir="", MODEL_CLASS=SiameseModel)

In [11]:
new_data['sbert_predicted'] = new_data.apply(lambda x: pipeline.get_prediction(x.name)[1], axis=1)
new_data['pre_sbert_predicted'] = new_data.apply(lambda x: pretrained_pipeline.get_prediction(x.name)[1], axis=1)

In [12]:
results = []

for col in new_data.columns[3:]:
    mse = mean_squared_error(new_data['normalized_score'], new_data[col])
    rmse = np.sqrt(mse)
    pearson_corr, _ = pearsonr(new_data['normalized_score'], new_data[col])
    
    # Append results as a dictionary
    results.append({'Predicted': col, 'dataset': 'indo synthetic', 'MSE': mse, 'RMSE': rmse, 'Pearson Correlation': pearson_corr})

results_df_synt = pd.DataFrame(results).set_index('Predicted')
results_df_synt.head(10)

,dataset,MSE,RMSE,Pearson Correlation
Predicted,,,,
sbert_predicted,indo synthetic,0.116008,0.34060,0.767120
pre_sbert_predicted,indo synthetic,0.256867,0.50682,0.681462


## RAHUMOTO + SAG

In [13]:
df_sag = pd.read_csv("../../../data/aes_dataset_unseen.csv")
print(df_sag.info())
df_sag.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 445 entries, 0 to 444
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   question          445 non-null    object 
 1   reference_answer  445 non-null    object 
 2   answer            445 non-null    object 
 3   score             445 non-null    float64
 4   normalized_score  445 non-null    float64
 5   dataset           445 non-null    object 
 6   dataset_num       445 non-null    object 
dtypes: float64(2), object(5)
memory usage: 24.5+ KB
None


,question,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [14]:
transformers.utils.logging.set_verbosity_error()
model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
model_dir = "../../../experiments/models/best_model/sbert_cos_sim1.pt"
pipeline = inference(df_sag, model_name, model_dir, SiameseModel)
pretrained_pipeline = inference(df_sag, model_name, model_dir="", MODEL_CLASS=SiameseModel)

In [15]:
df_sag['sbert_predicted'] = df_sag.apply(lambda x: pipeline.get_prediction(x.name)[1], axis=1)
df_sag['pre_sbert_predicted'] = df_sag.apply(lambda x: pretrained_pipeline.get_prediction(x.name)[1], axis=1)

In [16]:
results = []

for col in df_sag.columns[7:]:
    mse = mean_squared_error(df_sag['normalized_score'], df_sag[col])
    rmse = np.sqrt(mse)
    pearson_corr, _ = pearsonr(df_sag['normalized_score'], df_sag[col])
    
    # Append results as a dictionary
    results.append({'Predicted': col, 'dataset': 'indo & sag unseen', 'MSE': mse, 'RMSE': rmse, 'Pearson Correlation': pearson_corr})

results_df_mix = pd.DataFrame(results).set_index('Predicted')
results_df_mix.head(10)

,dataset,MSE,RMSE,Pearson Correlation
Predicted,,,,
sbert_predicted,indo & sag unseen,0.078487,0.280156,0.533331
pre_sbert_predicted,indo & sag unseen,0.126154,0.355181,0.243503


## CONCAT

In [17]:
concat_df = pd.concat([results_df.head(1), results_df_synt.head(1), results_df_mix.head(1)], axis=0)
concat_df = concat_df.reset_index()
concat_df = concat_df.set_index('dataset')
concat_df.head()

,Predicted,MSE,RMSE,Pearson Correlation
dataset,,,,
indo unseen,sbert_predicted,0.073067,0.270308,0.646366
indo synthetic,sbert_predicted,0.116008,0.340600,0.767120
indo & sag unseen,sbert_predicted,0.078487,0.280156,0.533331
